# KrishiNexus · `krishi-bangla-lora-v4`: QLoRA fine-tune on Kaggle (SFT prompts v7)

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA on the KrishiNexus Bangla agri-advisor SFT set, prompts
v7: farmers' questions are freely worded (colloquial Bangla, regional dialects, Banglish, English), each storage
question asks about its own answer sentence's subject, and the store-or-sell tool result states whether today's
price is below the production-cost floor. It then scores the
base model and the adapter on the **frozen test split**, exports a **Q8_0 GGUF + Ollama Modelfile** for the union
kiosk, and scores that GGUF through **Ollama**, the runtime the app uses. The recipe follows `TRAINING_PLAN.md`
sections 4–6 in the [KrishiNexus repo](https://github.com/HTanvir0011/KrishiNexus); section 6, "Prompts v7", has the
targets set before this run.

The test split holds a subset no earlier model saw the like of: `ood_wording`, questions in wordings written separately
from the training ones. It is scored like every other subset.

**Before you run**
1. **Settings → Accelerator → GPU T4 ×2.** The notebook uses one T4, and switches to bf16 on its own if you pick an L4 or a newer GPU.
2. **Settings → Internet → on.** `pip`, the model download and the Ollama install need it.
3. **Add Input** → the *KrishiNexus Bangla Agri-Advisor SFT v7* dataset. Without it, the notebook rebuilds the data from GitHub.
4. For an unattended run, use **Save Version → Save & Run All**. For a first smoke run, set `MAX_STEPS = 20` and
   `EVAL_LIMIT = 32` in the config cell. Every cell, including the export, then runs in about 30–40 minutes.

**Time (estimated, not yet measured):** training takes about 1.5–2.5 h on one T4, evaluation about 20–40 min, the export
about 15–25 min and the Ollama scoring about 20–40 min.

**Frozen-test rule:** training and checkpoint selection read only `train` + `val`. Section 3 opens `test` once, after
the best checkpoint is fixed. Never tune anything against it.

**Outputs (`/kaggle/working`):**
- `krishi-bangla-lora-v4/`: the adapter, its tokenizer and `eval_report.json`
- `krishi-bangla-q8_0.gguf` and `Modelfile`
- `gguf_eval_report.json` and `predictions_test_gguf.jsonl`: the GGUF scored through Ollama
- `predictions_test.jsonl`
- `training_log.json` and `loss_curve.png`
- `run_manifest.json`


In [ ]:
# Run this cell first: CUDA_VISIBLE_DEVICES only takes effect before torch is imported.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one T4 is plenty for 1.5B in 4-bit, and avoids DataParallel over a 4-bit model
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"           # Apache-2.0, ungated (TRAINING_PLAN.md section 3)
ADAPTER_NAME = "krishi-bangla-lora-v4"              # v2 (prompts v5) is deployed; v3 (prompts v6) missed one target
SEED = 20261022
EPOCHS, LEARNING_RATE = 3, 2e-4
MAX_STEPS = None                                    # e.g. 20 to push every cell through quickly; None trains all EPOCHS
BATCH, GRAD_ACCUM = 4, 4                            # effective batch 16, as in lora_adapter_spec
MAX_LEN = 1024                                      # the plan's 512 was for a 4 GB card; section 1 prints what each would cut
EVAL_LIMIT = None                                   # None = the whole frozen test split; e.g. 64 for a smoke run
GEN_BATCH = 32
RUN_BASE_EVAL = True                                # also score the untouched base model on identical prompts
EXPORT_GGUF = True                                  # merge into the de-quantised NF4 base -> GGUF Q8_0 + Ollama Modelfile
SCORE_GGUF = True                                   # score that GGUF through Ollama on the same test split
KEEP_INTERMEDIATES = False                          # keep the ~3 GB merged model, the f16 GGUF and the checkpoints
REPO_URL = "https://github.com/HTanvir0011/KrishiNexus.git"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("outputs")
os.makedirs(WORK, exist_ok=True)


In [ ]:
# Needs Internet on. Kaggle ships torch with CUDA; this adds the training stack at the versions the recipe was checked against.
%pip install -q -U "transformers>=5.22,<6" "trl>=1.17,<2" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.50.2"
# Kaggle ships torchao 0.10; PEFT refuses any torchao older than 0.16 when it merges, and nothing here uses it.
%pip uninstall -y -q torchao


In [ ]:
import gc, json, math, shutil, subprocess, sys, time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import torch

assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2, then run all again."
assert torch.cuda.device_count() == 1, "More than one GPU is visible: restart the session so the first cell's CUDA_VISIBLE_DEVICES applies."
CAPABILITY = torch.cuda.get_device_capability(0)
assert CAPABILITY >= (7, 0), (f"{torch.cuda.get_device_name(0)} is sm_{CAPABILITY[0]}{CAPABILITY[1]}; current bitsandbytes 4-bit "
                              "wheels need a newer GPU. Pick GPU T4 x2.")
BF16 = CAPABILITY >= (8, 0)                         # L4 / A100 / H100 have bf16; the T4 (sm_75) does not, so it trains in fp16
DTYPE = torch.bfloat16 if BF16 else torch.float16
VERSIONS = {p: version(p) for p in ("torch", "transformers", "trl", "peft", "accelerate", "datasets", "bitsandbytes")}
GPU = torch.cuda.get_device_name(0)
print(GPU, f"sm_{CAPABILITY[0]}{CAPABILITY[1]}", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB",
      "| precision:", "bf16" if BF16 else "fp16", "| python", sys.version.split()[0])
print(VERSIONS)
torch.manual_seed(SEED)


In [ ]:
# Data checks, deterministic scorers (no LLM judge, TRAINING_PLAN.md section 6) and the report. Pure Python, no GPU.
import json
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

SPLITS = ("train", "val", "test")
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
VERDICTS = ("BLOCK_SPOT_SALE", "ALLOW_STORAGE", "ALLOW_SPOT_SALE", "WARN_STORAGE_LOSES_VALUE", "WARN_NO_STORAGE_CAPACITY")
ALERTS = {"RED": "RED", "ORANGE": "ORANGE", "YELLOW": "YELLOW", "GREEN": "GREEN",
          "লাল": "RED", "কমলা": "ORANGE", "হলুদ": "YELLOW", "সবুজ": "GREEN"}
PRICE_RE = re.compile(r"খামার দাম কেজি (\d+) টাকা, পাইকারি (\d+) টাকা, খুচরা (\d+) টাকা")
TRACE_RE = re.compile(r"storage = [^;]*?= (-?[\d,]+); spot = [^;]*?= (-?[\d,]+); dPi = (-?[\d,]+)")
PCT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%")
CITE_RE = re.compile(r"\[([A-Z][A-Z0-9]*(?:-[A-Z0-9]+)+)\]")
UNFAIR = ("দামের চেয়ে", "ন্যায্য নয়", "ন্যায্য না", "অন্যায্য", "বিক্রি না করে", "বিক্রি করবেন না", "not fair", "unfair", "too low")
FAIR = ("ন্যায্য", "বিক্রি করতে পারেন", "fair")
TARGETS = {  # TRAINING_PLAN.md section 6
    "negotiation_decision_accuracy": 0.90, "negotiation_price_fidelity": 0.85,
    "arbitrage_verdict_exact_match": 0.90, "arbitrage_trace_fidelity": 0.80,
    "flood_alert_accuracy": 0.90, "flood_probability_within_2pts": 0.85,
    "storage_citation_accuracy": 0.90, "storage_chrf": 60.0,
}
LOWER_IS_BETTER = {"negotiation_unparsed_rate", "flood_probability_mae_pts"}


def find_data_dir():
    """An attached Kaggle dataset (any mount layout) or a local training/data folder; None if neither exists."""
    candidates = sorted(Path("/kaggle/input").glob("**/split_report.json")) if Path("/kaggle/input").exists() else []
    candidates += [Path(p) / "split_report.json" for p in ("training/data", "../training/data", "data")]
    for report in candidates:
        if report.exists() and all((report.parent / f"{s}.jsonl").exists() for s in SPLITS):
            return report.parent
    return None


def load_splits(data_dir):
    """Load the three JSONL splits and fail loudly on anything that would make training or scoring wrong."""
    data_dir = Path(data_dir)
    rows = {s: [json.loads(line) for line in (data_dir / f"{s}.jsonl").open(encoding="utf-8") if line.strip()] for s in SPLITS}
    report = json.loads((data_dir / "split_report.json").read_text(encoding="utf-8"))
    for split, n in report["splits"].items():
        assert len(rows[split]) == n, f"{split}.jsonl has {len(rows[split])} rows but split_report.json says {n}"
    ids = {s: {r["example_id"] for r in rows[s]} for s in SPLITS}
    assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"]), "an example_id is in two splits"
    # Answer families (identical category + answer) never cross splits; negotiation is split by market cell upstream.
    families = {s: {(r["category"], answer(r)) for r in rows[s] if r["category"] != "market_negotiation"} for s in SPLITS}
    assert not (families["train"] & families["val"] or families["train"] & families["test"] or families["val"] & families["test"]), \
        "an answer family straddles splits"
    for split in SPLITS:
        for r in rows[split]:
            assert [m["role"] for m in r["messages"]] == ["system", "user", "assistant"], f"{r['example_id']}: unexpected roles"
    return rows, report


def check_reference_formats(rows):
    """Every reference must parse, or the scorers below would silently mis-score."""
    for r in rows:
        ref, task = answer(r), r["category"]
        ok = {"market_negotiation": lambda: decision(ref) and PRICE_RE.search(ref),
              "arbitrage_math": lambda: verdict(ref) and trace(ref),
              "flood_harvest": lambda: alert(ref) and first_pct(ref) is not None,
              "storage_qa": lambda: len(citations(ref)) == 1}[task]()
        assert ok, f"{r['example_id']}: reference is not in the expected {task} format"


def answer(row):
    return row["messages"][-1]["content"]


def question(row):
    """The farmer's words without the tool result (prompts v4 append one), which a prompt-blind baseline may not read."""
    return row["messages"][1]["content"].split("\n<tool_result")[0]


def to_prompt_completion(row):
    """TRL conversational prompt-completion row: loss is computed on the assistant turn only."""
    return {"prompt": row["messages"][:-1], "completion": row["messages"][-1:]}


def token_lengths(tokenizer, rows):
    return [len(tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize=False), add_special_tokens=False)["input_ids"])
            for r in rows]


def eval_subset(rows, limit):
    """Deterministic stride sample that keeps every task (the file is ordered by task); all rows when limit is None."""
    if not limit or limit >= len(rows):
        return rows
    return rows[:: len(rows) // limit][:limit]


def norm(text):
    return text.translate(BN_DIGITS)


def ints(text):
    return {int(x) for x in re.findall(r"\d+", norm(text))}


def earliest(text, vocab):
    hits = [(text.find(v), v) for v in vocab if v in text]
    return min(hits)[1] if hits else None


def decision(text):
    t = norm(text).lower()
    return "unfair" if any(c in t for c in UNFAIR) else "fair" if any(c in t for c in FAIR) else None


def verdict(text):
    return earliest(text, VERDICTS)


def alert(text):
    hit = earliest(text, tuple(ALERTS))
    return ALERTS[hit] if hit else None


def first_pct(text):
    match = PCT_RE.search(norm(text))
    return float(match.group(1)) if match else None


def trace(text):
    match = TRACE_RE.search(norm(text))
    return tuple(int(v.replace(",", "")) for v in match.groups()) if match else None


def citations(text):
    return set(CITE_RE.findall(text))


def uncited(text):
    return CITE_RE.sub("", text).strip()


def close(got, gold):
    return abs(got - gold) <= max(1.0, 0.005 * abs(gold))


def chrf(hyps, refs, order=6, beta=2.0):
    """Corpus chrF2 as in sacreBLEU's default: char n-grams 1-6, whitespace removed, P and R averaged over orders."""
    stats = [[0, 0, 0] for _ in range(order)]  # hypothesis n-grams, reference n-grams, matches
    for hyp, ref in zip(hyps, refs):
        hyp, ref = "".join(hyp.split()), "".join(ref.split())
        for n in range(1, order + 1):
            h = Counter(hyp[i:i + n] for i in range(len(hyp) - n + 1))
            r = Counter(ref[i:i + n] for i in range(len(ref) - n + 1))
            stats[n - 1][0] += sum(h.values())
            stats[n - 1][1] += sum(r.values())
            stats[n - 1][2] += sum((h & r).values())
    eps = 1e-16
    precision = sum(m / h if h else eps for h, _, m in stats) / order
    recall = sum(m / r if r else eps for _, r, m in stats) / order
    factor = beta ** 2
    return 100 * (1 + factor) * precision * recall / (factor * precision + recall) if precision + recall else 0.0


def score_example(row, pred):
    """Per-example 0/1 scores for the row's task, keyed by report metric name."""
    ref, task = answer(row), row["category"]
    if task == "market_negotiation":
        return {"negotiation_decision_accuracy": decision(pred) == decision(ref),
                "negotiation_price_fidelity": set(map(int, PRICE_RE.search(ref).groups())) <= ints(pred)}
    if task == "arbitrage_math":
        got, gold = trace(pred), trace(ref)
        return {"arbitrage_verdict_exact_match": verdict(pred) == verdict(ref),
                "arbitrage_trace_fidelity": got is not None and all(close(a, b) for a, b in zip(got, gold))}
    if task == "flood_harvest":
        got = first_pct(pred)
        return {"flood_alert_accuracy": alert(pred) == alert(ref),
                "flood_probability_within_2pts": got is not None and abs(got - first_pct(ref)) <= 2}
    return {"storage_citation_accuracy": citations(pred) == citations(ref)}


def aggregate(rows, preds):
    """metric -> {value, n} for 'all' and each test subset; chrF, balanced accuracy and MAE are computed per group."""
    out = {}
    for group in ["all"] + sorted({r["subset"] for r in rows}):
        sel = [(r, p) for r, p in zip(rows, preds) if group == "all" or r["subset"] == group]
        sums, counts = defaultdict(float), Counter()
        for r, p in sel:
            for metric, value in score_example(r, p).items():
                sums[metric] += float(value)
                counts[metric] += 1
        res = {m: {"value": round(sums[m] / counts[m], 4), "n": counts[m]} for m in sorted(counts)}
        pairs = [(decision(answer(r)), decision(p)) for r, p in sel if r["category"] == "market_negotiation"]
        if pairs:
            recalls = []
            for label in ("fair", "unfair"):
                gold_n = sum(g == label for g, _ in pairs)
                if gold_n:
                    recalls.append(sum(g == label and q == label for g, q in pairs) / gold_n)
            res["negotiation_balanced_accuracy"] = {"value": round(sum(recalls) / len(recalls), 4), "n": len(pairs)}
            res["negotiation_unparsed_rate"] = {"value": round(sum(q is None for _, q in pairs) / len(pairs), 4), "n": len(pairs)}
        errors = [abs(first_pct(p) - first_pct(answer(r))) for r, p in sel
                  if r["category"] == "flood_harvest" and first_pct(p) is not None]
        if any(r["category"] == "flood_harvest" for r, _ in sel):
            res["flood_probability_mae_pts"] = {"value": round(statistics.mean(errors), 2) if errors else None, "n": len(errors)}
        storage = [(uncited(p), uncited(answer(r))) for r, p in sel if r["category"] == "storage_qa"]
        if storage:
            res["storage_chrf"] = {"value": round(chrf(*zip(*storage)), 2), "n": len(storage)}
        out[group] = res
    return out


def baseline_predictions(test_rows, train_rows):
    """Prompt-blind baselines built from TRAIN only: the majority label per task, and a train-answer lookup for storage."""
    def majority(parse, task):
        return Counter(parse(answer(r)) for r in train_rows if r["category"] == task).most_common(1)[0][0]

    negotiation = {"unfair": "ন্যায্য খামার দামের চেয়ে কম।", "fair": "এই দাম ন্যায্য, বিক্রি করতে পারেন।"}[majority(decision, "market_negotiation")]
    arbitrage, flood = majority(verdict, "arbitrage_math"), f"({majority(alert, 'flood_harvest')})"
    by_question = defaultdict(list)
    for r in train_rows:
        if r["category"] == "storage_qa":
            by_question[question(r)].append(answer(r))

    def storage(question):
        options = by_question.get(question)
        if not options:
            return ""
        doc = Counter(next(iter(citations(a))) for a in options).most_common(1)[0][0]
        return next(a for a in options if doc in citations(a))

    fixed = {"market_negotiation": negotiation, "arbitrage_math": arbitrage, "flood_harvest": flood}
    return [fixed.get(r["category"]) or storage(question(r)) for r in test_rows]


def build_report(test_rows, preds, train_rows, meta):
    """preds = {"adapter": [...], "base": [...]} in test_rows order; adds the prompt-blind baseline, targets and deltas."""
    results = {name: aggregate(test_rows, p) for name, p in preds.items()}
    results["baseline"] = aggregate(test_rows, baseline_predictions(test_rows, train_rows))

    def overall(name, metric):
        return results.get(name, {}).get("all", {}).get(metric, {}).get("value")

    summary = []
    for metric in sorted(results["baseline"]["all"]):
        a, b, z = overall("adapter", metric), overall("base", metric), overall("baseline", metric)
        better = None if a is None or b is None else (a < b if metric in LOWER_IS_BETTER else a > b)
        summary.append({"metric": metric, "target": TARGETS.get(metric), "adapter": a, "base": b, "baseline": z,
                        "improved_over_base": better,
                        "meets_target": None if metric not in TARGETS or a is None else a >= TARGETS[metric]})
    return {**meta, "targets": TARGETS, "summary": summary, "results": results}


def _fmt(value):
    return "–" if value is None else f"{value:.3f}" if isinstance(value, float) else str(value)


def summary_table(report):
    lines = ["| metric | target | adapter | base | baseline | adapter beats base | meets target |", "|---|---|---|---|---|---|---|"]
    for s in report["summary"]:
        name = s["metric"] + (" (lower is better)" if s["metric"] in LOWER_IS_BETTER else "")
        lines.append(f"| {name} | {_fmt(s['target'])} | {_fmt(s['adapter'])} | {_fmt(s['base'])} | {_fmt(s['baseline'])} "
                     f"| {_fmt(s['improved_over_base'])} | {_fmt(s['meets_target'])} |")
    return "\n".join(lines)


def subset_table(report, model="adapter"):
    groups = [g for g in report["results"][model] if g != "all"]
    lines = [f"| {model}: metric | " + " | ".join(groups) + " |", "|---|" + "---|" * len(groups)]
    for metric in sorted(report["results"][model]["all"]):
        cells = [report["results"][model][g].get(metric, {}) for g in groups]
        lines.append(f"| {metric} | " + " | ".join(f"{_fmt(c.get('value'))} (n={c['n']})" if c else "–" for c in cells) + " |")
    return "\n".join(lines)


## 1 · Data

This section finds the attached dataset (any `/kaggle/input` layout), or rebuilds it from GitHub, and then checks it:
- the split counts match `split_report.json`;
- no example or answer family appears in two splits;
- every reference answer parses, so the scorers can't silently mis-score.

These integrity checks are the only place this section reads the test split.


In [ ]:
DATA = find_data_dir()
if DATA is None:
    print("No attached dataset with train/val/test.jsonl + split_report.json found; rebuilding it from GitHub.")
    repo = Path(WORK) / "KrishiNexus"
    try:
        if not repo.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic>=2", "httpx", "numpy", "python-dotenv"], check=True)
        subprocess.run([sys.executable, str(repo / "training" / "prepare_data.py")], check=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError("Rebuilding the data failed (Internet off, or a Python older than the 3.12 the backend is "
                           "checked on). Attach the dataset instead: Add Input > Datasets.") from error
    DATA = repo / "training" / "data"
ROWS, SPLIT_REPORT = load_splits(DATA)
for split in SPLITS:
    check_reference_formats(ROWS[split])
SYSTEM_PROMPT = SPLIT_REPORT["system_prompt"]
print("data:", DATA, "| sha256", SPLIT_REPORT.get("jsonl_sha256", "?")[:12])
for split in SPLITS:
    print(f"{split:5s} {len(ROWS[split]):5d}", dict(Counter(r["category"] for r in ROWS[split])))
print("test subsets:", dict(Counter(r["subset"] for r in ROWS["test"])))


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
# TRL adds no EOS to chat data: the template must close the answer, or the model never learns to stop.
assert tok.apply_chat_template(ROWS["train"][0]["messages"], tokenize=False).rstrip().endswith("<|im_end|>"), \
    "the chat template does not end the assistant turn with <|im_end|>"
lengths = {split: token_lengths(tok, ROWS[split]) for split in ("train", "val")}
for split, ls in lengths.items():
    print(f"{split}: longest {max(ls)} tokens, p99 {sorted(ls)[int(0.99 * len(ls))]} | would truncate at 512: "
          f"{sum(n > 512 for n in ls)}, at MAX_LEN={MAX_LEN}: {sum(n > MAX_LEN for n in ls)}")
assert all(n <= MAX_LEN for ls in lengths.values() for n in ls), "Raise MAX_LEN: some answers would lose their ending."
train_ds = Dataset.from_list([to_prompt_completion(r) for r in ROWS["train"]])
val_ds = Dataset.from_list([to_prompt_completion(r) for r in ROWS["val"]])
print(train_ds, val_ds, sep="\n")


## 2 · Train

QLoRA as specified in `lora_adapter_spec`:
- a frozen 4-bit NF4 base;
- rank-16 LoRA on all 7 projection matrices;
- completion-only loss;
- 3 epochs, effective batch 16, learning rate 2e-4 on a cosine schedule.

The best checkpoint is picked by **validation** loss, never by test.


In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=DTYPE, attn_implementation="sdpa", device_map={"": 0},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True))
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
total_steps = MAX_STEPS or math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM)) * EPOCHS
args = SFTConfig(
    output_dir=f"{WORK}/checkpoints", max_length=MAX_LEN, num_train_epochs=EPOCHS, max_steps=MAX_STEPS or -1,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=8, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_steps=max(1, round(0.03 * total_steps)),
    optim="paged_adamw_8bit", bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps", eval_steps=50, save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=10, dataloader_num_workers=0, report_to="none", seed=SEED)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tok, peft_config=peft_config)
if not BF16:
    # TRL casts the QLoRA adapter weights to bf16; the T4 has no bf16, so keep them fp32 under fp16 mixed precision.
    for param in trainer.model.parameters():
        if param.requires_grad:
            param.data = param.data.float()
trainer.model.print_trainable_parameters()
print(f"{total_steps} optimizer steps planned")


In [ ]:
started = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - started) / 60, 1)
ADAPTER_DIR = Path(WORK) / ADAPTER_NAME
trainer.save_model(str(ADAPTER_DIR))                # the best checkpoint by validation loss (load_best_model_at_end)
tok.save_pretrained(str(ADAPTER_DIR))
(Path(WORK) / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=1), encoding="utf-8")
best = trainer.state.best_metric
print(f"trained in {TRAIN_MINUTES} min | best checkpoint {trainer.state.best_model_checkpoint} | "
      f"best val loss {best:.4f}" if best is not None else f"trained in {TRAIN_MINUTES} min (no evaluation ran)")


In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in log if "loss" in h]
val_pts = [(h["step"], h["eval_loss"]) for h in log if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if train_pts:
    plt.plot(*zip(*train_pts), label="train loss")
if val_pts:
    plt.plot(*zip(*val_pts), "o-", label="validation loss")
plt.xlabel("optimizer step")
plt.ylabel("loss on answer tokens")
plt.title(f"{ADAPTER_NAME} on {BASE_MODEL}")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(f"{WORK}/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 3 · Evaluate on the frozen test split

**Decoding.** Greedy (no sampling, no repetition penalty), with identical prompts for both models. The base model is
the same 4-bit weights with the adapter switched off.

**Scoring.** Deterministic, with no LLM judge. Each answer is parsed and compared with the engine-written reference on
the decision, verdict, alert, numbers, citation and chrF. Results are reported for the whole set and for each
held-out family.

**Baseline column.** A **prompt-blind** predictor built from train only: the majority label per task, or the train
answer to the same storage question. Read every target against it, because a target this baseline also meets says
nothing about the model.


In [ ]:
from contextlib import nullcontext
from transformers import GenerationConfig

gc.collect()
torch.cuda.empty_cache()
TEST = eval_subset(ROWS["test"], EVAL_LIMIT)        # first and only time the test answers are used
model = trainer.model
model.eval()
model.config.use_cache = True
tok.padding_side = "left"                           # batched generation pads on the left
max_new = min(512, max(len(tok(answer(r), add_special_tokens=False)["input_ids"]) for r in TEST) + 32)
gen_config = GenerationConfig(do_sample=False, max_new_tokens=max_new, repetition_penalty=1.0, use_cache=True,
                              pad_token_id=tok.pad_token_id,
                              eos_token_id=[tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")])


def generate(rows, use_adapter):
    prompts = [tok.apply_chat_template(r["messages"][:-1], tokenize=False, add_generation_prompt=True) for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [""] * len(prompts)
    with (nullcontext() if use_adapter else model.disable_adapter()), torch.inference_mode():
        for start in range(0, len(order), GEN_BATCH):
            idx = order[start:start + GEN_BATCH]
            enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            new = model.generate(**enc, generation_config=gen_config)[:, enc["input_ids"].shape[1]:]
            for i, seq in zip(idx, new):
                out[i] = tok.decode(seq, skip_special_tokens=True).strip()
    return out


PREDS = {}
for name, use_adapter in [("adapter", True)] + ([("base", False)] if RUN_BASE_EVAL else []):
    t0 = time.time()
    PREDS[name] = generate(TEST, use_adapter)
    print(f"{name}: {len(TEST)} answers in {(time.time() - t0) / 60:.1f} min")
with open(f"{WORK}/predictions_test.jsonl", "w", encoding="utf-8") as fh:
    for i, r in enumerate(TEST):
        fh.write(json.dumps({"example_id": r["example_id"], "category": r["category"], "subset": r["subset"],
                             "dialect": r["dialect"], "question": r["messages"][1]["content"], "reference": answer(r),
                             **{name: p[i] for name, p in PREDS.items()}}, ensure_ascii=False) + "\n")


In [ ]:
from datetime import datetime, timezone
from IPython.display import Markdown, display

REPORT = build_report(TEST, PREDS, ROWS["train"], meta={
    "adapter": ADAPTER_NAME, "base_model": BASE_MODEL, "status": "trained",
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "split": "test", "n_examples": len(TEST), "eval_limit": EVAL_LIMIT, "data_sha256": SPLIT_REPORT.get("jsonl_sha256"),
    "prompt_version": SPLIT_REPORT.get("prompt_version", 4),
    "decoding": {"do_sample": False, "max_new_tokens": max_new, "repetition_penalty": 1.0, "weights": "4-bit NF4 base, as trained"},
    "gpu": GPU, "precision": "bf16" if BF16 else "fp16", "versions": VERSIONS, "train_minutes": TRAIN_MINUTES,
    "best_val_loss": trainer.state.best_metric, "max_len": MAX_LEN,
    "truncated_train_examples": sum(n > MAX_LEN for n in lengths["train"])})
for path in (Path(WORK) / "eval_report.json", ADAPTER_DIR / "eval_report.json"):
    path.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
display(Markdown(summary_table(REPORT)))
display(Markdown(subset_table(REPORT, "adapter")))


## 4 · Export for Ollama

Ollama ≥ 0.34.2 no longer loads LoRA adapters, so the adapter has to be merged into the model. This section:
1. merges the adapter into the base it was trained against: the 4-bit NF4 weights, de-quantised to 16-bit;
2. converts the result straight to an 8-bit Q8_0 GGUF with llama.cpp.

Why: a merge into the full-precision bf16 base, quantised to Q4_K_M, lost flood-alert and store-or-sell decisions on
the test split. This export reproduced the training run's scores (TRAINING_PLAN.md section 5).

The llama.cpp converter pins CPU torch and transformers 4.x. It is installed into a side folder, so the training stack
stays untouched.


In [ ]:
from peft import PeftModel

del trainer, model
gc.collect()
torch.cuda.empty_cache()
MERGED_DIR = Path(WORK) / "merged"
GGUF_ERROR = None                                   # Kaggle discards every output of a failed run, so export errors are caught
if EXPORT_GGUF:
    try:
        # The adapter learned against the NF4 weights, so it is merged into exactly those, de-quantised.
        nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=nf4, dtype=DTYPE, device_map={"": 0})
        base = base.dequantize() or base             # some transformers versions de-quantise in place
        if hasattr(base.config, "quantization_config"):
            del base.config.quantization_config
        merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR)).merge_and_unload()
        merged.save_pretrained(str(MERGED_DIR))
        # Training never changes the tokenizer. Ship Qwen's original files: llama.cpp's pinned transformers 4.x can't
        # parse a tokenizer_config.json re-saved by transformers 5.
        from huggingface_hub import snapshot_download
        snapshot_download(BASE_MODEL, local_dir=str(MERGED_DIR), allow_patterns=["tokenizer*", "vocab.json", "merges.txt"])
        del base, merged
        gc.collect()
        print("merged ->", MERGED_DIR)
    except Exception as error:
        GGUF_ERROR = f"merge: {error!r}"
        print("Merge failed:", GGUF_ERROR)


In [ ]:
def run(cmd, **kwargs):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kwargs)


def export_gguf():
    llama = Path(WORK) / "llama.cpp"
    if not llama.exists():
        run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", llama])
    deps = Path(WORK) / "gguf-deps"                 # the converter's pinned torch/transformers live here, not in the kernel
    run([sys.executable, "-m", "pip", "install", "-q", "--target", deps, "-r",
         llama / "requirements" / "requirements-convert_hf_to_gguf.txt"])
    run([sys.executable, llama / "convert_hf_to_gguf.py", MERGED_DIR, "--outtype", "q8_0", "--outfile", GGUF_Q8],
        env={**os.environ, "PYTHONPATH": str(deps)})
    (Path(WORK) / "Modelfile").write_text(f"FROM ./{GGUF_Q8.name}\nPARAMETER temperature 0.2\nSYSTEM {SYSTEM_PROMPT}\n",
                                          encoding="utf-8")
    print("GGUF ->", GGUF_Q8, f"{GGUF_Q8.stat().st_size / 2**30:.2f} GiB")


GGUF_Q8 = Path(WORK) / "krishi-bangla-q8_0.gguf"  # the file name lora_adapter_spec's Modelfile expects
if EXPORT_GGUF and GGUF_ERROR is None:
    try:
        export_gguf()
    except Exception as error:  # Kaggle discards every output of a failed run, so an export error must not fail it
        GGUF_ERROR = repr(error)
        print(f"GGUF export failed: {GGUF_ERROR}\nThe adapter, eval report and predictions are saved; convert later per "
              "TRAINING_PLAN.md section 5.")
if not KEEP_INTERMEDIATES:
    for path in (MERGED_DIR, Path(WORK) / "llama.cpp", Path(WORK) / "gguf-deps", Path(WORK) / "checkpoints"):
        if path.is_dir():
            shutil.rmtree(path, ignore_errors=True)
        else:
            path.unlink(missing_ok=True)


## 5 · Score the GGUF through Ollama

The app runs the model through Ollama, so the exported file is scored the way the app calls it: `/api/generate` with
each test question as the prompt, the Modelfile's SYSTEM line and greedy decoding. The report,
`gguf_eval_report.json`, uses the same scorer and the same test rows as section 3.


In [ ]:
import traceback
import urllib.request

OLLAMA, GGUF_SCORES, GGUF_SCORE_ERROR = "http://127.0.0.1:11434", None, None


def ollama(path, body=None, timeout=600):
    req = urllib.request.Request(OLLAMA + path, data=None if body is None else json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


if SCORE_GGUF and GGUF_Q8.exists():
    try:   # Kaggle discards every output of a failed run, so a scoring error is caught and recorded
        # Ollama ships as .tar.zst and Kaggle's image has no zstd to unpack it.
        run(["bash", "-c", "(apt-get update -qq || true) && apt-get install -y -qq zstd && curl -fsSL https://ollama.com/install.sh | sh"])
        server = subprocess.Popen(["ollama", "serve"], stdout=open(Path(WORK) / "ollama.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(90):
            try:
                OLLAMA_VERSION = ollama("/api/version")["version"]
                break
            except OSError:
                time.sleep(1)
        run(["ollama", "create", "krishi-bangla", "-f", Path(WORK) / "Modelfile"], cwd=WORK)
        gguf_preds, rates, started = [], [], time.time()
        for row in TEST:
            out = ollama("/api/generate", {"model": "krishi-bangla", "prompt": row["messages"][1]["content"], "stream": False,
                                           "options": {"temperature": 0, "num_predict": max_new, "seed": 0}})
            gguf_preds.append(out["response"])
            if out.get("eval_duration"):
                rates.append(out["eval_count"] / (out["eval_duration"] / 1e9))
        GGUF_REPORT = build_report(TEST, {"adapter": gguf_preds}, ROWS["train"], {
            "export": GGUF_Q8.name, "runtime": f"Ollama {OLLAMA_VERSION}", "gguf_bytes": GGUF_Q8.stat().st_size,
            "split": "test", "n_examples": len(TEST), "data_sha256": SPLIT_REPORT.get("jsonl_sha256"),
            "prompt_version": SPLIT_REPORT.get("prompt_version", 4),
            "decoding": {"temperature": 0, "num_predict": max_new, "seed": 0, "system": "Modelfile SYSTEM line"},
            "minutes": round((time.time() - started) / 60, 1),
            "decode_tokens_per_s_median": round(sorted(rates)[len(rates) // 2], 1) if rates else None,
            "identical_to_4bit_answers": sum(p.strip() == q.strip() for p, q in zip(gguf_preds, PREDS["adapter"])),
            "note": "'adapter' in the summary is this GGUF served by Ollama on the T4"})
        (Path(WORK) / "gguf_eval_report.json").write_text(json.dumps(GGUF_REPORT, ensure_ascii=False, indent=2), encoding="utf-8")
        with open(Path(WORK) / "predictions_test_gguf.jsonl", "w", encoding="utf-8") as fh:
            for r, p in zip(TEST, gguf_preds):
                fh.write(json.dumps({"example_id": r["example_id"], "category": r["category"], "subset": r["subset"],
                                     "gguf": p}, ensure_ascii=False) + "\n")
        GGUF_SCORES = {s["metric"]: s["adapter"] for s in GGUF_REPORT["summary"]}
        display(Markdown(summary_table(GGUF_REPORT)))
        display(Markdown(subset_table(GGUF_REPORT, "adapter")))
        server.terminate()
    except Exception:
        GGUF_SCORE_ERROR = traceback.format_exc()
        print(GGUF_SCORE_ERROR[-1500:])


In [ ]:
manifest = {"adapter": ADAPTER_NAME, "base_model": BASE_MODEL, "seed": SEED, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
            "effective_batch": BATCH * GRAD_ACCUM, "max_len": MAX_LEN, "precision": "bf16" if BF16 else "fp16", "gpu": GPU,
            "versions": VERSIONS, "data_dir": str(DATA), "data_sha256": SPLIT_REPORT.get("jsonl_sha256"),
            "prompt_version": SPLIT_REPORT.get("prompt_version", 4),
            "splits": SPLIT_REPORT["splits"], "train_minutes": TRAIN_MINUTES, "eval_examples": len(TEST),
            "gguf_exported": GGUF_Q8.exists(), "gguf_error": GGUF_ERROR,
            "gguf_scores": GGUF_SCORES, "gguf_score_error": GGUF_SCORE_ERROR}
(Path(WORK) / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
for path in sorted(Path(WORK).iterdir()):
    size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path.is_dir() else path.stat().st_size
    print(f"{size / 2**20:9.1f} MiB  {path.name}")


## 6 · Bring it home

Only if the pre-registered targets in TRAINING_PLAN.md section 6 ("Prompts v7") are met:
1. Download these from the version's **Output** tab: `krishi-bangla-lora-v4/`, `krishi-bangla-q8_0.gguf`, `Modelfile`,
   `gguf_eval_report.json` and the two predictions files.
2. Copy `krishi-bangla-lora-v4/` to `backend/models/lora/` and the GGUF report to `backend/models/lora/gguf_eval_report.json`.
3. In the folder with the GGUF, run `ollama create krishi-bangla -f Modelfile`.
4. Set `PROMPT_VERSION = 7` and the registry's `lora_adapter` to `krishi-bangla-lora-v4` in
   `backend/edge_node/ollama_local.py`, so the router adds `spot_below_floor` to store-or-sell tool results.
5. Record the measured numbers from both reports in `MEMORY.md`, and never type them into the app (AGENTS.md §2).
